# 05 — Sonuçların Analizi ve Karşılaştırma

**Amaç:**
- Tüm deney sonuçlarını birleştir ve görselleştir
- Confusion matrix ve ROC eğrilerini oluştur
- Literatür ile karşılaştırma tablosu hazırla
- Makale için hazır figürler üret

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import full_preprocessing_pipeline
from src.evaluation import (
    evaluate_model, plot_confusion_matrix,
    plot_roc_curves, plot_feature_importance,
    compare_models_table
)

DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'
METRICS_DIR = '../results/metrics/'
FIG_DIR = '../results/figures/'
FIG_KWARGS = dict(dpi=150, bbox_inches='tight')

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='Set2')

## 1. Eğitilmiş Modelleri ve Test Verisini Yükle

In [ ]:
X_train, X_test, y_train, y_test, tld_encoder, scaler = full_preprocessing_pipeline(DATA_PATH)
feature_names = list(X_train.columns)

feature_sets = joblib.load(f'{METRICS_DIR}feature_sets.joblib')
vajrobol_5 = feature_sets['vajrobol_5']
rf_top20 = feature_sets['rf_top20']

trained_exp1 = joblib.load(f'{METRICS_DIR}trained_models_exp1.joblib')
trained_exp2 = joblib.load(f'{METRICS_DIR}trained_models_exp2.joblib')
trained_exp3 = joblib.load(f'{METRICS_DIR}trained_models_exp3.joblib')

print('Modeller ve veri yüklendi.')

## 2. Deney Sonuçlarını Yükle

In [ ]:
df_exp1 = pd.read_csv(f'{METRICS_DIR}exp1_full_features.csv', index_col=0)
df_exp2 = pd.read_csv(f'{METRICS_DIR}exp2_vajrobol5.csv', index_col=0)
df_exp3 = pd.read_csv(f'{METRICS_DIR}exp3_rf_top20.csv', index_col=0)
df_cv   = pd.read_csv(f'{METRICS_DIR}exp4_cross_validation.csv', index_col=0)

print('Deney 1 — Tam Özellik Seti:')
print(df_exp1[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].to_string())

## 3. Confusion Matrix — Deney 1 Tüm Modeller

In [ ]:
for model_name, model in trained_exp1.items():
    y_pred = model.predict(X_test)
    save_path = f'{FIG_DIR}cm_exp1_{model_name.replace(" ", "_")}.png'
    plot_confusion_matrix(y_test, y_pred, f'{model_name} (Tam Özellik)', save_path)

## 4. ROC Eğrileri — Tüm Deneyler

In [ ]:
plot_roc_curves(trained_exp1, X_test, y_test, f'{FIG_DIR}roc_exp1_full.png')

X_test_v5 = X_test[vajrobol_5]
plot_roc_curves(trained_exp2, X_test_v5, y_test, f'{FIG_DIR}roc_exp2_vajrobol5.png')

X_test_rf20 = X_test[rf_top20]
plot_roc_curves(trained_exp3, X_test_rf20, y_test, f'{FIG_DIR}roc_exp3_rf_top20.png')

## 5. Özellik Önemi — RF ve XGBoost

In [ ]:
plot_feature_importance(
    trained_exp1['Random Forest'], feature_names,
    top_n=20, save_path=f'{FIG_DIR}fi_rf_full.png'
)

plot_feature_importance(
    trained_exp1['XGBoost'], feature_names,
    top_n=20, save_path=f'{FIG_DIR}fi_xgb_full.png'
)

## 6. Karşılaştırmalı Barchart — Tüm Deneyler × Metrikler

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(22, 5))

for ax, metric in zip(axes, metrics_to_plot):
    if metric not in df_exp1.columns:
        continue
    x = np.arange(len(df_exp1.index))
    w = 0.25
    exp1_vals = df_exp1[metric].fillna(0).values
    exp2_vals = df_exp2[metric].reindex(df_exp1.index).fillna(0).values
    exp3_vals = df_exp3[metric].reindex(df_exp1.index).fillna(0).values

    ax.bar(x - w, exp1_vals, w, label='Exp1: Tam', color='steelblue')
    ax.bar(x, exp2_vals, w, label='Exp2: Vajrobol5', color='tomato')
    ax.bar(x + w, exp3_vals, w, label='Exp3: RF Top20', color='seagreen')

    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ', '\n') for m in df_exp1.index], fontsize=7)
    ax.set_ylim(0.85, 1.02)
    ax.set_title(metric, fontsize=10)
    ax.legend(fontsize=6)
    ax.axhline(y=0.97, color='red', linestyle='--', linewidth=0.8, alpha=0.7)

plt.suptitle('Deney × Metrik Karşılaştırması', fontsize=13)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}experiment_comparison.png', **FIG_KWARGS)
plt.show()

## 7. Literatür ile Karşılaştırma Tablosu

In [ ]:
# En iyi modelimizin sonuçları
best_model_name = df_exp1['F1-Score'].idxmax()
best_metrics = df_exp1.loc[best_model_name]
vajrobol_lr_acc = df_exp2.loc['Logistic Regression', 'Accuracy'] if 'Logistic Regression' in df_exp2.index else None

literature = pd.DataFrame([
    {'Kaynak': 'Prasad & Chandra (2024)',  'Yöntem': 'BernoulliNB+PAC+SGD',      'Veri Seti': 'PhiUSIIL',    'Accuracy': 0.9924, 'F1': 0.9921, 'Recall': None},
    {'Kaynak': 'Vajrobol vd. (2024)',      'Yöntem': 'LR (5 özellik)',           'Veri Seti': 'PhiUSIIL',    'Accuracy': 0.9997, 'F1': 0.9997, 'Recall': None},
    {'Kaynak': 'Yoon vd. (2024)',          'Yöntem': 'CNN+Transformer+GCN',      'Veri Seti': 'Common Crawl','Accuracy': 0.9812, 'F1': 0.9789, 'Recall': None},
    {'Kaynak': 'Rao vd. (2025)',           'Yöntem': 'Super Learner Ensemble',   'Veri Seti': 'PhishDump',   'Accuracy': 0.9893, 'F1': None,   'Recall': None},
    {'Kaynak': 'Taha vd. (2024)',          'Yöntem': 'LR,DT,RF,XGB',            'Veri Seti': '11K Phishing','Accuracy': 0.9689, 'F1': None,   'Recall': None},
    {'Kaynak': f'Bu Çalışma — {best_model_name} (Exp1)', 'Yöntem': best_model_name, 'Veri Seti': 'PhiUSIIL',
     'Accuracy': best_metrics.get('Accuracy'), 'F1': best_metrics.get('F1-Score'), 'Recall': best_metrics.get('Recall')},
    {'Kaynak': 'Bu Çalışma — LR (Vajrobol-5)',  'Yöntem': 'LR (5 özellik)', 'Veri Seti': 'PhiUSIIL',
     'Accuracy': vajrobol_lr_acc, 'F1': df_exp2.loc['Logistic Regression', 'F1-Score'] if 'Logistic Regression' in df_exp2.index else None, 'Recall': None},
])

literature = literature.set_index('Kaynak')
print(literature.to_string())
literature.to_csv(f'{METRICS_DIR}literature_comparison.csv')
print('\nKarşılaştırma tablosu kaydedildi.')

## 8. Özet Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sol üst: Deney 1 Accuracy
ax = axes[0, 0]
df_exp1_sorted = df_exp1.sort_values('Accuracy')
ax.barh(df_exp1_sorted.index, df_exp1_sorted['Accuracy'], color='steelblue')
ax.set_xlim(0.85, 1.0)
ax.axvline(x=0.97, color='red', linestyle='--', label='Hedef: 0.97')
ax.set_title('Deney 1: Accuracy (Tam Özellik)')
ax.legend()

# Sağ üst: F1-Score karşılaştırması
ax = axes[0, 1]
models = df_exp1.index
x = np.arange(len(models))
ax.plot(x, df_exp1['F1-Score'].values, 'o-', label='Exp1: Tam', color='steelblue')
ax.plot(x, df_exp2['F1-Score'].reindex(models).values, 's-', label='Exp2: Vajrobol5', color='tomato')
ax.plot(x, df_exp3['F1-Score'].reindex(models).values, '^-', label='Exp3: RF Top20', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=8)
ax.axhline(y=0.96, color='red', linestyle='--', linewidth=0.8)
ax.set_title('F1-Score — Deney Karşılaştırması')
ax.legend()
ax.set_ylim(0.85, 1.02)

# Sol alt: Recall karşılaştırması
ax = axes[1, 0]
ax.plot(x, df_exp1['Recall'].values, 'o-', label='Exp1: Tam', color='steelblue')
ax.plot(x, df_exp2['Recall'].reindex(models).values, 's-', label='Exp2: Vajrobol5', color='tomato')
ax.set_xticks(x)
ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=8)
ax.axhline(y=0.97, color='red', linestyle='--', linewidth=0.8, label='Hedef: 0.97')
ax.set_title('Recall (Odak Metrik) — Deney Karşılaştırması')
ax.legend()
ax.set_ylim(0.85, 1.02)

# Sağ alt: CV sonuçları
ax = axes[1, 1]
if not df_cv.empty:
    cv_acc_cols = [c for c in df_cv.columns if 'accuracy_mean' in c]
    if cv_acc_cols:
        ax.barh(df_cv.index, df_cv[cv_acc_cols[0]], color='mediumpurple')
        ax.set_title('Deney 4: 10-Fold CV Accuracy')
        ax.set_xlim(0.9, 1.0)

plt.suptitle('Phishing URL Tespiti — Sonuç Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}results_dashboard.png', **FIG_KWARGS)
plt.show()
print('Dashboard kaydedildi.')

## Sonuç

Bu notebook'ta:
- 4 farklı deney için confusion matrix, ROC eğrileri ve metrik tabloları oluşturuldu
- Özellik önemi görselleştirildi
- Literatür ile karşılaştırma tablosu hazırlandı
- Makale için kullanılabilir figürler `results/figures/` klasörüne kaydedildi

Tüm sayısal sonuçlar `results/metrics/` klasöründe CSV formatında mevcut.